# 🏙️ UrbanHub — Jumeau Numérique Urbain
### Tableau de bord de supervision Smart City (Restitution Gold)

> **Architecture :** Lecture seule sur la couche `gold` du Data Lake S3 (MinIO)  
> **Sources :** Météo NOAA, Mobilité CityBikes, Pollution OpenAQ v3

In [ ]:
import os
import io
import json
import yaml
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from minio import Minio
from minio.error import S3Error

# 1. Chargement de la configuration des villes cibles
config_path = "../config/cities.yml" if os.path.exists("../config/cities.yml") else "config/cities.yml"
with open(config_path, "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

cities_dict = {c["name"]: c["id"] for c in config.get("cities", [])}

# 2. Connexion au Data Lake (Sécurisé lecture seule sur Gold)
MINIO_ENDPOINT = os.getenv("MINIO_ENDPOINT", "localhost:9000")
client = Minio(
    MINIO_ENDPOINT,
    access_key=os.getenv("MINIO_ACCESS_KEY", "minioadmin"),
    secret_key=os.getenv("MINIO_SECRET_KEY", "minioadmin123"),
    secure=False
)
print(f"✅ Connecté au Data Lake : {MINIO_ENDPOINT}")

## 🎛️ Sélecteurs de Supervision

In [ ]:
# Widgets de sélection
city_selector = widgets.Dropdown(
    options=list(cities_dict.keys()),
    value=list(cities_dict.keys())[0] if cities_dict else None,
    description="Ville :",
    layout=widgets.Layout(width="300px")
)

refresh_btn = widgets.Button(
    description="Actualiser les KPI",
    button_style="info",
    icon="sync",
    layout=widgets.Layout(width="180px")
)

output_kpi = widgets.Output()

display(widgets.HBox([city_selector, refresh_btn]))
display(output_kpi)

## 📊 Vues Métier par Flux

In [ ]:
def render_dashboard(b=None):
    selected_city_name = city_selector.value
    selected_city_id = cities_dict.get(selected_city_name)
    
    with output_kpi:
        clear_output(wait=True)
        display(HTML(f"<h3>Statistiques pour l'agglomération : <span style='color:#2980b9;'>{selected_city_name} ({selected_city_id})</span></h3>"))
        
        # Création des 4 onglets thématiques
        tab = widgets.Tab()
        tab_titles = ["🌦️ Météo (NOAA)", "🚲 Mobilité (CityBikes)", "🌫️ Pollution (OpenAQ)", "🔄 Analyse Croisée"]
        
        # --- Onglet 1 : Météo ---
        out_weather = widgets.Output()
        with out_weather:
            try:
                # Tentative de lecture de la table Gold
                resp = client.get_object("gold", "kpis/weather_daily.parquet")
                df_w = pd.read_parquet(io.BytesIO(resp.read()))
                resp.close()
                resp.release_conn()
                display(df_w.head())
            except Exception:
                display(HTML("<p style='color:#e67e22;'>⏳ <b>En attente des données Gold Météo de Rémi...</b><br>Table attendue : <code>gold/kpis/weather_daily.parquet</code></p>"))
        
        # --- Onglet 2 : Mobilité ---
        out_mobility = widgets.Output()
        with out_mobility:
            try:
                resp = client.get_object("gold", "kpis/mobility_station_latest.parquet")
                df_m = pd.read_parquet(io.BytesIO(resp.read()))
                resp.close()
                resp.release_conn()
                display(df_m.head())
            except Exception:
                display(HTML("<p style='color:#e67e22;'>⏳ <b>En attente des données Gold Mobilité de Conambot...</b><br>Table attendue : <code>gold/kpis/mobility_station_latest.parquet</code></p>"))
        
        # --- Onglet 3 : Pollution ---
        out_pollution = widgets.Output()
        with out_pollution:
            try:
                resp = client.get_object("gold", "kpis/pollution_hourly.parquet")
                df_p = pd.read_parquet(io.BytesIO(resp.read()))
                resp.close()
                resp.release_conn()
                display(df_p.head())
            except Exception:
                display(HTML("<p style='color:#e67e22;'>⏳ <b>En attente des données Gold Pollution de Julie...</b><br>Table attendue : <code>gold/kpis/pollution_hourly.parquet</code></p>"))
        
        # --- Onglet 4 : Analyse Croisée ---
        out_cross = widgets.Output()
        with out_cross:
            try:
                resp = client.get_object("gold", "kpis/city_hourly.parquet")
                df_c = pd.read_parquet(io.BytesIO(resp.read()))
                resp.close()
                resp.release_conn()
                display(df_c.head())
            except Exception:
                display(HTML("<p style='color:#e67e22;'>⏳ <b>En attente de la table croisée multi-sources...</b><br>Table attendue : <code>gold/kpis/city_hourly.parquet</code></p>"))
        
        tab.children = [out_weather, out_mobility, out_pollution, out_cross]
        for i, title in enumerate(tab_titles):
            tab.set_title(i, title)
        display(tab)

refresh_btn.on_click(render_dashboard)
render_dashboard()